# Land Use Change Detection

## 📊 Business Context
Monitor urban expansion.

**Analytical Approach:** Change Detection
This notebook utilizes advanced analytics to derive actionable insights.

In [ ]:
# Import Libraries
import ee
import geemap
import matplotlib.pyplot as plt

# Initialize Earth Engine
try:
    ee.Initialize()
except:
    ee.Authenticate()
    ee.Initialize()

In [ ]:
# Define AOI and Time Periods
AOI = ee.Geometry.Point([3.3792, 6.5244]).buffer(20000) # Lagos

def detect_changes():
    print('Processing satellite imagery...')
    # Load Image Collection (Sentinel-2 or MODIS)
    # Using MODIS for broader temporal coverage in this demo
    dataset = ee.ImageCollection('COPERNICUS/S2')
    
    # Period 1 (Baseline)
    img1 = dataset.filterDate('2018-01-01', '2018-12-31') \
                  .filterBounds(AOI) \
                  .select('B8') \
                  .mean().clip(AOI)
                  
    # Period 2 (Comparison)
    img2 = dataset.filterDate('2023-01-01', '2023-12-31') \
                  .filterBounds(AOI) \
                  .select('B8') \
                  .mean().clip(AOI)
    
    # Calculate Difference
    diff = img2.subtract(img1)
    
    # Thresholding for significant change (e.g., > 10% change)
    # Adjust threshold based on data range
    threshold = 1000 # Example for scaled NDVI or LST
    significant_increase = diff.gt(threshold)
    significant_decrease = diff.lt(-threshold)
    
    # Calculate Change Area
    pixel_area = ee.Image.pixelArea()
    increase_area = significant_increase.multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=AOI, scale=500, maxPixels=1e9
    ).get('B8').getInfo() / 1e6
    
    decrease_area = significant_decrease.multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=AOI, scale=500, maxPixels=1e9
    ).get('B8').getInfo() / 1e6
    
    print(f'Significant Increase Area: {increase_area:.2f} sq km')
    print(f'Significant Decrease Area: {decrease_area:.2f} sq km')
    
    # Visualization
    m = geemap.Map(center=[6.5244, 3.3792], zoom=10)
    
    vis_params = {'min': 0, 'max': 3000}
    diff_vis = {'min': -2000, 'max': 2000, 'palette': ['blue', 'white', 'red']}
    
    m.addLayer(img1, vis_params, 'Baseline (2018)')
    m.addLayer(img2, vis_params, 'Comparison (2023)')
    m.addLayer(diff, diff_vis, 'Difference Map')
    m.addLayer(significant_increase.updateMask(significant_increase), {'palette': ['red']}, 'Significant Increase')
    
    m.add_colorbar(diff_vis, label='Change Magnitude')
    return m

m = detect_changes()
m

## 📉 Change Analysis

1. **Trend Direction**: The analysis reveals a net [increase/decrease] in `B8` over the 5-year period.
2. **Spatial Pattern**: Changes are concentrated in [Urban/Rural] zones, suggesting...
3. **Implication**: This shift impacts [Temperature/Vegetation/Urbanization] dynamics, requiring...